In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()
        self.Flatten = nn.Flatten()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)

        x = self.Flatten(x)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [3]:
def check_shape():
    model = LeNet5()
    x = torch.randn(1, 1, 28, 28)
    print(f"输入: {x.shape}")
    # 逐层检查
    x1 = model.conv1(x)
    print(f"Conv1(1->6, k=5, p=2): {x1.shape}  # 期望 (1,6,28,28)")
    x2 = model.relu(x1)
    print(f"ReLU: {x2.shape}")
    x3 = model.pool1(x2)
    print(f"Pool1(k=2, s=2): {x3.shape}  # 期望 (1,6,14,14)")
    x4 = model.conv2(x3)
    print(f"Conv2(6->16, k=5, p=0): {x4.shape}  # 期望 (1,16,10,10)")
    x5 = model.relu(x4)
    print(f"ReLU: {x5.shape}")
    x6 = model.pool2(x5)
    print(f"Pool2(k=2, s=2): {x6.shape}  # 期望 (1,16,5,5)")
    x7 = x6.view(x6.size(0), -1)
    print(f"Flatten: {x7.shape}  # 期望 (1,400)")
    x8 = model.fc1(x7)
    print(f"FC1(400->120): {x8.shape}")
    x9 = model.relu(x8)
    print(f"ReLU: {x9.shape}")
    x10 = model.fc2(x9)
    print(f"FC2(120->84): {x10.shape}")
    x11 = model.relu(x10)
    print(f"ReLU: {x11.shape}")
    x12 = model.fc3(x11)
    print(f"FC3(84->10): {x12.shape}")
    print("\n断言验证...")
    assert x1.shape == (1, 6, 28, 28), f"Conv1输出错误: {x1.shape}"
    assert x3.shape == (1, 6, 14, 14), f"Pool1输出错误: {x3.shape}"
    assert x4.shape == (1, 16, 10, 10), f"Conv2输出错误: {x4.shape}"
    assert x6.shape == (1, 16, 5, 5), f"Pool2输出错误: {x6.shape}"
    assert x7.shape == (1, 400), f"Flatten输出错误: {x7.shape}"
    assert x8.shape == (1, 120), f"FC1输出错误: {x8.shape}"
    assert x10.shape == (1, 84), f"FC2输出错误: {x10.shape}"
    assert x12.shape == (1, 10), f"FC3输出错误: {x12.shape}"

In [4]:
def train_and_evaluate(num_epochs=10, batch_size=64):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"使用设备: {device}")

    # 数据预处理
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 加载 FashionMNIST 数据集
    train_dataset = torchvision.datasets.FashionMNIST(
        root='./data', train=True, download=True, transform=transform
    )
    test_dataset = torchvision.datasets.FashionMNIST(
        root='./data', train=False, download=True, transform=transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 初始化模型、损失函数、优化器
    model = LeNet5().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    print("开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_dataset)

        # 测试阶段
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    print("训练完成！")
    return model

In [5]:
check_shape()
trained_model = train_and_evaluate(num_epochs=10, batch_size=64)

输入: torch.Size([1, 1, 28, 28])
Conv1(1->6, k=5, p=2): torch.Size([1, 6, 28, 28])  # 期望 (1,6,28,28)
ReLU: torch.Size([1, 6, 28, 28])
Pool1(k=2, s=2): torch.Size([1, 6, 14, 14])  # 期望 (1,6,14,14)
Conv2(6->16, k=5, p=0): torch.Size([1, 16, 10, 10])  # 期望 (1,16,10,10)
ReLU: torch.Size([1, 16, 10, 10])
Pool2(k=2, s=2): torch.Size([1, 16, 5, 5])  # 期望 (1,16,5,5)
Flatten: torch.Size([1, 400])  # 期望 (1,400)
FC1(400->120): torch.Size([1, 120])
ReLU: torch.Size([1, 120])
FC2(120->84): torch.Size([1, 84])
ReLU: torch.Size([1, 84])
FC3(84->10): torch.Size([1, 10])

断言验证...
使用设备: cpu


100%|██████████| 26.4M/26.4M [00:05<00:00, 4.75MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 129kB/s]
100%|██████████| 4.42M/4.42M [00:10<00:00, 423kB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.25MB/s]


开始训练...
Epoch [1/10] | Loss: 0.5917 | Test Accuracy: 84.91%
Epoch [2/10] | Loss: 0.3685 | Test Accuracy: 86.77%
Epoch [3/10] | Loss: 0.3210 | Test Accuracy: 87.51%
Epoch [4/10] | Loss: 0.2967 | Test Accuracy: 88.73%
Epoch [5/10] | Loss: 0.2730 | Test Accuracy: 88.92%
Epoch [6/10] | Loss: 0.2545 | Test Accuracy: 90.00%
Epoch [7/10] | Loss: 0.2428 | Test Accuracy: 89.49%
Epoch [8/10] | Loss: 0.2289 | Test Accuracy: 89.84%
Epoch [9/10] | Loss: 0.2179 | Test Accuracy: 89.89%
Epoch [10/10] | Loss: 0.2092 | Test Accuracy: 90.34%
训练完成！
